In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

In [4]:
MYTH_ORDER  = ["clothing", "victim_intoxication", "perpetrator_intoxication", "resistance"]
MYTH_LABELS = {
    "clothing":                 "Clothing",
    "victim_intoxication":      "Victim\nIntox.",
    "perpetrator_intoxication": "Perp.\nIntox.",
    "resistance":               "Resistance",
}
MODELS = ["gemma", "llama", "phi", "qwen", "mistral"]
EMBEDDING   = "sbert"   # primary embedding for figures; others available in CSV

Path("Figures").mkdir(parents=True, exist_ok=True)

In [6]:
SHIFT_DIR  = Path("/mnt/beegfs/msaxena4/6_Explicit-Implicit-Bias/ResultAnalysis/Summarization/SampleResults")
FIG_DIR    = SHIFT_DIR / "Figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
 
sns.set_theme(style="whitegrid", font_scale=1.15)
MODEL_PALETTE = sns.color_palette("Set2", n_colors=len(MODELS))
MODEL_COLORS  = dict(zip(MODELS, MODEL_PALETTE))
MYTH_PALETTE  = sns.color_palette("muted", n_colors=len(MYTH_ORDER))
MYTH_COLORS   = dict(zip(MYTH_ORDER, MYTH_PALETTE))
 

In [7]:
def sig_stars(p_bh: float) -> str:
    if p_bh < 0.001: return "***"
    if p_bh < 0.01:  return "**"
    if p_bh < 0.05:  return "*"
    return ""

In [8]:
def save(fig, name: str):
    path = FIG_DIR / name
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {path.name}")

In [10]:
# ── Load all model results ────────────────────────────────────────────────────
append_dfs   = {}
original_dfs = {}

for model in MODELS:
    ap = SHIFT_DIR / f"1_SemanticShift/{model}_Append_Embeddings.csv"
    og = SHIFT_DIR / f"1_SemanticShift/{model}_Original_Embeddings.csv"
    if ap.exists():
        df = pd.read_csv(ap)
        df["model"] = model
        append_dfs[model] = df
    else:
        print(f"  WARNING: {ap.name} not found, skipping {model} append")
    if og.exists():
        df = pd.read_csv(og)
        df["model"] = model
        original_dfs[model] = df
    else:
        print(f"  WARNING: {og.name} not found, skipping {model} original")

In [11]:
# ── Save summary CSVs ─────────────────────────────────────────────────────────
if append_dfs:
    combined_append = pd.concat(append_dfs.values(), ignore_index=True)
    # combined_append.to_csv(FIG_DIR / "stat_summary_append.csv", index=False)
    # print(f"  Saved: stat_summary_append.csv")

if original_dfs:
    combined_original = pd.concat(original_dfs.values(), ignore_index=True)
    # combined_original.to_csv(FIG_DIR / "stat_summary_original.csv", index=False)
    # print(f"  Saved: stat_summary_original.csv")

In [46]:
table = all_append[
    (all_append["embedding"] == EMBEDDING) &
    (all_append["is_pair"] == False) &
    (all_append["myth"].isin(MYTH_ORDER)) &
    (all_append["significant"] == True)
].copy()

In [53]:
# # ── Reporting tables for ContextAppend ───────────────────────────────────────
# if append_dfs:
#     all_append = pd.concat(append_dfs.values(), ignore_index=True)
#     table = all_append[
#         (all_append["embedding"] == EMBEDDING) &
#         (all_append["is_pair"] == False) &
#         (all_append["myth"].isin(MYTH_ORDER)) &
#         (all_append["significant"] == True)
#     ].copy()

#     table["mean_cosine_similarity"] = (1 - table["mean_cosine_distance"]).round(4)
#     table["cohens_dz"]  = table["cohens_dz"].round(4)
#     table["p_bh"]      = table["p_bh"]

#     table = table[[
#         "model", "myth", "n",
#         "mean_cosine_similarity", "cohens_dz",
#         "test", "p_bh",
#     ]].sort_values(["model", "cohens_dz"], ascending=[True, True])

#     out = FIG_DIR / f"StatisticalTest_ContextAppend.csv"
#     table.to_csv(out, index=False)

In [59]:
# all_append[
#     (all_append["embedding"] == EMBEDDING) &
#     (all_append["is_pair"] == True) &
#     (all_append["significant"] == True)
# ]

In [60]:
"""
4_Tables_SemanticShift.py
=========================
Generate reporting tables for 1c_SemanticShift outputs.

Table 1 — ContextAppend: Semantic Shift (SBERT)
  Filter: embedding == "sbert"
  Columns: model, myth, is_pair, test, n, mean_cosine_similarity, cohens_dz, p, p_bh, significant

Table 2 — Original: Effect Size by Myth (SBERT)
  Filter: embedding == "sbert"
  Columns: model, myth_type, test, n_entail, n_contradict, cohens_d, p, p_bh, significant

Usage
-----
  python 4_Tables_SemanticShift.py \\
      --shift_dir path/to/1_SemanticShift/ \\
      --models    gemma llama phi qwen mistral \\
      --task      Summarization
"""

import argparse
import pandas as pd
from pathlib import Path

# ── Args ──────────────────────────────────────────────────────────────────────
parser = argparse.ArgumentParser()
parser.add_argument("--shift_dir", required=True,
                    help="Path to 1_SemanticShift/ directory")
parser.add_argument("--models", nargs="+",
                    default=["gemma", "llama", "phi", "qwen", "mistral"])
parser.add_argument("--task", choices=["Advice", "Summarization", "AdviceGeneration"],
                    default="Summarization")
args = parser.parse_args()



In [ ]:
SHIFT_DIR = Path("/mnt/beegfs/msaxena4/6_Explicit-Implicit-Bias/ResultAnalysis/Summarization/SampleResults/1_SemanticShift")
OUT_DIR   = SHIFT_DIR / "Figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MYTH_ORDER = ["clothing", "victim_intoxication", "perpetrator_intoxication", "resistance"]

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
append_dfs   = []
original_dfs = []

for model in args.models:
    ap = SHIFT_DIR / f"{model}_Append_Embeddings.csv"
    og = SHIFT_DIR / f"{model}_Original_Embeddings.csv"

    if ap.exists():
        df = pd.read_csv(ap)
        df["model"] = model
        append_dfs.append(df)
    else:
        print(f"  WARNING: {ap.name} not found, skipping")

    if og.exists():
        df = pd.read_csv(og)
        df["model"] = model
        original_dfs.append(df)
    else:
        print(f"  WARNING: {og.name} not found, skipping")

    print()

In [ ]:
# ── Table 1: ContextAppend ────────────────────────────────────────────────────
if append_dfs:
    df = pd.concat(append_dfs, ignore_index=True)
    t1 = df[df["embedding"] == "sbert"].copy()

    t1["mean_cosine_similarity"] = (1 - t1["mean_cosine_distance"]).round(4)
    t1["cohens_dz"]  = t1["cohens_dz"].round(4)
    t1["p"]          = t1["p"].round(4)
    t1["p_bh"]       = t1["p_bh"].round(4)

    t1 = t1[[
        "model", "myth", "is_pair", "test",
        "n", "mean_cosine_similarity", "cohens_dz",
        "p", "p_bh", "significant",
    ]].sort_values(["model", "is_pair", "myth"]).reset_index(drop=True)

    out = OUT_DIR / f"Table1_ContextAppend_SemanticShift_{args.task}.csv"
    t1.to_csv(out, index=False)
    print(f"Saved: {out.name}")
    print(t1.to_string(index=False))


In [ ]:
# ── Table 2: Original ─────────────────────────────────────────────────────────
if original_dfs:
    df = pd.concat(original_dfs, ignore_index=True)
    t2 = df[df["embedding"] == "sbert"].copy()

    t2["cohens_d"] = t2["cohens_d"].round(4)
    t2["p"]        = t2["p"].round(4)
    t2["p_bh"]     = t2["p_bh"].round(4)

    t2 = t2[[
        "model", "myth_type", "test",
        "n_entail", "n_contradict", "cohens_d",
        "p", "p_bh", "significant",
    ]].sort_values(["model", "myth_type"]).reset_index(drop=True)

    out = OUT_DIR / f"Table2_Original_SemanticShift_{args.task}.csv"
    t2.to_csv(out, index=False)
    print(f"Saved: {out.name}")
    print(t2.to_string(index=False))

In [29]:
# # ── Per-model: ContextAppend cosine similarity barplot ───────────────────────
# for model, df in append_dfs.items():
#     sub = df[
#         (df["embedding"] == EMBEDDING) &
#         (df["is_pair"] == False) &
#         (df["myth"].isin(MYTH_ORDER))
#     ].copy()
#     if sub.empty:
#         print(f"  WARNING: no single-myth SBERT rows for {model} append, skipping")
#         continue

#     sub["myth_label"] = sub["myth"].map(MYTH_LABELS)
#     sub = sub.set_index("myth").reindex(MYTH_ORDER).reset_index()
#     sub["myth_label"] = sub["myth"].map(MYTH_LABELS)
#     sub["mean_cosine_similarity"] = 1 - sub["mean_cosine_distance"]

#     fig, ax = plt.subplots(figsize=(6, 3.5))
#     bars = ax.bar(
#         sub["myth_label"],
#         sub["mean_cosine_similarity"],
#         color=[MYTH_COLORS[m] for m in sub["myth"]],
#         alpha=0.8, edgecolor="none",
#     )
#     for bar, (_, row) in zip(bars, sub.iterrows()):
#         stars = sig_stars(row["p_bh"]) if pd.notna(row["p_bh"]) else ""
#         if stars:
#             ax.text(
#                 bar.get_x() + bar.get_width() / 2,
#                 bar.get_height() + 0.001,
#                 stars, ha="center", va="bottom", fontsize=11, color="#333333",
#             )
#     ax.set_xlabel("Myth Type", fontsize=10)
#     ax.set_ylabel("Mean Cosine Similarity (T1↔T2)", fontsize=10)
#     ax.set_title(f"{model.capitalize()} — Semantic Shift [Summarization]", fontsize=11)
#     ax.set_ylim(0.8, 1.02)
#     ax.yaxis.grid(True, linewidth=0.5, alpha=0.5)
#     ax.set_axisbelow(True)
#     sns.despine(ax=ax, left=False, bottom=False)
#     # save(fig, f"{model}_append_cosine_similarity_by_myth.png")

In [20]:
# # ── Per-model: Original Cohen's d barplot ────────────────────────────────────
# for model, df in original_dfs.items():
#     sub = df[
#         (df["embedding"] == EMBEDDING) &
#         (df["myth_type"].isin(MYTH_ORDER))
#     ].copy()

#     if sub.empty:
#         print(f"  WARNING: no SBERT rows for {model} original, skipping")
#         continue

#     sub["myth_label"] = sub["myth_type"].map(MYTH_LABELS)
#     sub = sub.set_index("myth_type").reindex(MYTH_ORDER).reset_index()
#     sub["myth_label"] = sub["myth_type"].map(MYTH_LABELS)

#     fig, ax = plt.subplots(figsize=(7, 4))
#     bars = ax.bar(
#         sub["myth_label"],
#         sub["cohens_d"],
#         color=[MYTH_COLORS[m] for m in sub["myth_type"]],
#         alpha=0.85, edgecolor="white", linewidth=0.8,
#     )
#     ax.axhline(0, color="black", linewidth=0.8, linestyle="-")
#     # Cohen's d thresholds
#     for thresh, label, ls in [(0.2, "small", ":"), (0.5, "medium", "--"), (0.8, "large", "-.")]:
#         ax.axhline(thresh,  color="gray", linewidth=0.8, linestyle=ls, alpha=0.5)
#         ax.axhline(-thresh, color="gray", linewidth=0.8, linestyle=ls, alpha=0.5)

#     for bar, (_, row) in zip(bars, sub.iterrows()):
#         stars = sig_stars(row["p_bh"]) if pd.notna(row["p_bh"]) else ""
#         if stars:
#             ypos = row["cohens_d"] + (0.02 if row["cohens_d"] >= 0 else -0.06)
#             ax.text(
#                 bar.get_x() + bar.get_width() / 2,
#                 ypos, stars, ha="center", va="bottom", fontsize=13, fontweight="bold",
#             )

#     ax.set_xlabel("Myth Type")
#     ax.set_ylabel("Cohen's d  (entailment vs neutral)")
#     ax.set_title(f"Original Experiment Effect Size — {model.capitalize()} [Summarization]")

#     # Legend for threshold lines
#     handles = [
#         mpatches.Patch(color="gray", alpha=0.0, label="d thresholds: 0.2 / 0.5 / 0.8")
#     ]
#     ax.legend(handles=handles, fontsize=8, loc="upper right")
#     # save(fig, f"{model}_original_cohens_d_by_myth.png")

In [30]:
# # ── Combined: cosine similarity heatmap myth × model ─────────────────────────
# if append_dfs:
#     all_append = pd.concat(append_dfs.values(), ignore_index=True)
#     sbert_append = all_append[
#         (all_append["embedding"] == EMBEDDING) &
#         (all_append["is_pair"] == False) &
#         (all_append["myth"].isin(MYTH_ORDER))
#     ]
#     sbert_append = sbert_append.copy()
#     sbert_append["mean_cosine_similarity"] = 1 - sbert_append["mean_cosine_distance"]
#     pivot = sbert_append.pivot_table(
#         index="myth", columns="model",
#         values="mean_cosine_similarity", aggfunc="mean"
#     ).reindex(index=MYTH_ORDER, columns=MODELS)

#     fig, ax = plt.subplots(figsize=(max(6, len(MODELS) * 1.4), 3.5))
#     sns.heatmap(
#         pivot, annot=True, fmt=".3f", cmap="YlOrRd_r",
#         linewidths=0.4, linecolor="#eeeeee", ax=ax,
#         cbar_kws={"label": "Mean Cosine Similarity", "shrink": 0.8},
#         yticklabels=[MYTH_LABELS[m] for m in MYTH_ORDER],
#         vmin=0.8, vmax=1.0,
#     )
#     ax.set_title(f"Semantic Shift: Mean Cosine Similarity — Summarization", fontsize=11)
#     ax.set_xlabel("Model", fontsize=10)
#     ax.set_ylabel("")
#     ax.tick_params(axis="both", length=0)
#     save(fig, "all_models_append_cosine_similarity_heatmap.png")

In [31]:
# # ── Combined: Cohen's d heatmap myth × model ─────────────────────────────────
# if original_dfs:
#     all_original = pd.concat(original_dfs.values(), ignore_index=True)
#     sbert_original = all_original[
#         (all_original["embedding"] == EMBEDDING) &
#         (all_original["myth_type"].isin(MYTH_ORDER))
#     ]
#     pivot = sbert_original.pivot_table(
#         index="myth_type", columns="model",
#         values="cohens_d", aggfunc="mean"
#     ).reindex(index=MYTH_ORDER, columns=MODELS)

#     fig, ax = plt.subplots(figsize=(max(6, len(MODELS) * 1.4), 4))
#     sns.heatmap(
#         pivot, annot=True, fmt=".3f", cmap="RdBu_r", center=0,
#         linewidths=0.5, ax=ax,
#         cbar_kws={"label": "Cohen's d"},
#         yticklabels=[MYTH_LABELS[m] for m in MYTH_ORDER],
#     )
#     ax.set_title(f"Original Experiment: Cohen's d (entailment vs neutral) — Summarization")
#     ax.set_xlabel("Model")
#     ax.set_ylabel("Myth Type")
#     # save(fig, "all_models_original_cohens_d_heatmap.png")

In [32]:
# # ── Combined: Hedges' g heatmap myth × model (ContextAppend) ─────────────────
# if append_dfs:
#     all_append = pd.concat(append_dfs.values(), ignore_index=True)
#     sbert_append = all_append[
#         (all_append["embedding"] == EMBEDDING) &
#         (all_append["is_pair"] == False) &
#         (all_append["myth"].isin(MYTH_ORDER))
#     ]
#     pivot_g = sbert_append.pivot_table(
#         index="myth", columns="model",
#         values="hedges_g", aggfunc="mean"
#     ).reindex(index=MYTH_ORDER, columns=MODELS)

#     fig, ax = plt.subplots(figsize=(max(6, len(MODELS) * 1.4), 3.5))
#     sns.heatmap(
#         pivot_g, annot=True, fmt=".3f", cmap="RdBu_r", center=0,
#         linewidths=0.4, linecolor="#eeeeee", ax=ax,
#         cbar_kws={"label": "Hedges' g", "shrink": 0.8},
#         yticklabels=[MYTH_LABELS[m] for m in MYTH_ORDER],
#     )
#     ax.set_title(f"Semantic Shift: Hedges' g — Summarization", fontsize=11)
#     ax.set_xlabel("Model", fontsize=10)
#     ax.set_ylabel("")
#     ax.tick_params(axis="both", length=0)
#     save(fig, "all_models_append_hedges_g_heatmap.png")

In [33]:
# # ── Validity check: perpetrator intoxication vs others ───────────────────────
# if append_dfs:
#     all_append = pd.concat(append_dfs.values(), ignore_index=True)
#     sbert_append = all_append[
#         (all_append["embedding"] == EMBEDDING) &
#         (all_append["is_pair"] == False) &
#         (all_append["myth"].isin(MYTH_ORDER))
#     ]
#     agg = sbert_append.groupby(["model", "myth"])["mean_cosine_distance"].mean().reset_index()

#     fig, ax = plt.subplots(figsize=(9, 4))
#     x      = np.arange(len(MODELS))
#     width  = 0.18
#     offset = np.linspace(-(len(MYTH_ORDER)-1)/2, (len(MYTH_ORDER)-1)/2, len(MYTH_ORDER)) * width

#     for i, myth in enumerate(MYTH_ORDER):
#         vals = [
#             agg[(agg["model"] == m) & (agg["myth"] == myth)]["mean_cosine_distance"].values
#             for m in MODELS
#         ]
#         vals = [v[0] if len(v) > 0 else np.nan for v in vals]
#         hatch = "//" if myth == "perpetrator_intoxication" else None
#         ax.bar(
#             x + offset[i], vals, width,
#             label=MYTH_LABELS[myth].replace("\n", " "),
#             color=MYTH_COLORS[myth], alpha=0.85,
#             hatch=hatch, edgecolor="white",
#         )

#     ax.set_xticks(x)
#     ax.set_xticklabels([m.capitalize() for m in MODELS])
#     ax.set_ylabel("Mean Cosine Distance (T1→T2)")
#     ax.set_title(f"Validity Check: Perpetrator Intoxication vs Others — Summarization")
#     ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
#     ax.set_xlabel("Model")
#     # save(fig, "validity_check_perpetrator_intoxication.png")

# print(f"\nAll figures saved to: {FIG_DIR}")